In [1]:
import re
import time
import requests
import pandas as pd
from datetime import date, datetime
from dateutil.relativedelta import relativedelta

# NyTimes

### Setting up API

This is a simple test of the URL key

In [5]:
API_KEY = "PeqtKANF3zHPj8AF5IWpZtCaWDMKZpO9YOmNEMZvGk2JbPjA" # TODO : Move to .env file before pushing to GitHub

url_ny = "https://api.nytimes.com/svc/news/v3/content/nyt/world.json"


'''
    Simple request function
'''
def get_sample_data(url, api_key=API_KEY):
    headers = {
        'api-key': api_key
        }

    response = requests.request("GET", url, params=headers)
    if response.status_code == 200:
        return response.json()
    else:
        return f"Error: {response.status_code}"

response = get_sample_data(url_ny)
print(response.keys())

dict_keys(['status', 'copyright', 'num_results', 'results'])


### Retrive articles that is about trump

The API dones't support that we download all the content that from the article, so we need a workaround. Insted content consist of lead_paragraph, abstract, snippet wich are different leves of summery.


In [6]:
BASE_URL = "https://api.nytimes.com/svc/archive/v1/{year}/{month}.json"


def month_range_last_n_months(n=6):
    """Return a sorted list of (year, month) for the last n months, including current month."""
    start = date.today().replace(day=1)
    months = []

    for i in range(n):
        d = start - relativedelta(months=i)
        months.append((d.year, d.month))

    return sorted(months)


def fetch_archive_month(year, month, api_key, max_retries=3, sleep_between_retries=10):
    """
    Fetch one NYT archive month with simple retry logic.
    This helps avoid failing immediately if the API is temporarily unhappy.
    """
    url = BASE_URL.format(year=year, month=month)
    params = {"api-key": api_key}

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, params=params, timeout=30)

            if response.status_code == 200:
                return response.json()

            print(f"Request failed for {year}-{month:02d} "
                  f"(status {response.status_code}), attempt {attempt}/{max_retries}")

        except requests.RequestException as e:
            print(f"Request error for {year}-{month:02d}, attempt {attempt}/{max_retries}: {e}")

        if attempt < max_retries:
            time.sleep(sleep_between_retries)

    print(f"Skipping {year}-{month:02d} after {max_retries} failed attempts.")
    return {}


def article_mentions_trump(doc):
    """
    Return True if the article text appears to mention Trump.
    Uses word boundaries to reduce false matches.
    """
    headline = (doc.get("headline") or {}).get("main", "")
    text_parts = [
        headline,
        doc.get("abstract", ""),
        doc.get("snippet", ""),
        doc.get("lead_paragraph", "")
    ]

    combined_text = " ".join(part for part in text_parts if part).lower()

    patterns = [
        r"\bdonald trump\b",
        r"\bpresident trump\b",
        r"\btrump\b"
    ]

    return any(re.search(pattern, combined_text) for pattern in patterns)


def normalize_article(doc):
    """Extract the fields you want: date, title, content."""
    raw_date = doc.get("pub_date", "")
    try:
        clean_date = datetime.fromisoformat(raw_date.replace("Z", "+00:00")).date().isoformat()
    except Exception:
        clean_date = raw_date

    title = (doc.get("headline") or {}).get("main", "")

    content_parts = [
        doc.get("lead_paragraph", ""),
        doc.get("abstract", ""),
        doc.get("snippet", "")
    ]
    content = " ".join(part.strip() for part in content_parts if part)

    return {
        "date": clean_date,
        "title": title,
        "content": content
    }


def get_trump_articles_last_6_months(api_key, months=6, sleep_between_requests=12):
    """
    Pull NYT archive data month by month, slowly, to avoid too many requests at once.
    """
    rows = []
    months_to_fetch = month_range_last_n_months(months)

    for year, month in months_to_fetch:
        print(f"Fetching {year}-{month:02d} ...")

        data = fetch_archive_month(year, month, api_key)
        docs = data.get("response", {}).get("docs", [])

        print(f"  Retrieved {len(docs)} articles")

        for doc in docs:
            if article_mentions_trump(doc):
                rows.append(normalize_article(doc))

        print(f"  Waiting {sleep_between_requests} seconds before next request...")
        time.sleep(sleep_between_requests)

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.drop_duplicates(subset=["date", "title", "content"])
        df = df.sort_values("date").reset_index(drop=True)

    return df


# Run
df_trump = get_trump_articles_last_6_months(
    api_key=API_KEY,
    months=6,
    sleep_between_requests=12
)

print(df_trump.head())
print(f"\nFound {len(df_trump)} matching articles.")

# Save to CSV
df_trump.to_csv("nyt_trump_last_6_months.csv", index=False, encoding="utf-8")
print("\nSaved to nyt_trump_last_6_months.csv")

Fetching 2025-11 ...
  Retrieved 3759 articles
  Waiting 12 seconds before next request...
Fetching 2025-12 ...
  Retrieved 3569 articles
  Waiting 12 seconds before next request...
Fetching 2026-01 ...
  Retrieved 4046 articles
  Waiting 12 seconds before next request...
Fetching 2026-02 ...
  Retrieved 3929 articles
  Waiting 12 seconds before next request...
Fetching 2026-03 ...
  Retrieved 4637 articles
  Waiting 12 seconds before next request...
Fetching 2026-04 ...
  Retrieved 1634 articles
  Waiting 12 seconds before next request...
         date                                              title  \
0  2025-11-01  As the Shutdown Pain Grows, Trump Attends to O...   
1  2025-11-01  Trump Threatens Military Action in Nigeria Ove...   
2  2025-11-01  Trump Administration Must Make Food Stamp Paym...   
3  2025-11-01                A Third Trump Term Is Not the Charm   
4  2025-11-01  Can South Korea Manage the Competing Needs of ...   

                                             

# Fox news

Fox news doenes't have an API and I couldn't get the web scraping working. Super easily and don't want to spend more time on it.

# NEWS API

Deleted all the code again, since we can only get 1 month of data with NewsAPI using the free plan.

# GNews

Deleted all the code again, same issue as the NewsAPI where the free token is not enough for 6 month of data

# The guardian

In [14]:
import time
import requests
import pandas as pd

API_KEY = "89147664-d710-4997-b2c0-5bafc085c116" # TODO : Move to .env file before pushing to GitHub
BASE_URL = "https://content.guardianapis.com/search"


def fetch_guardian_page(
    query,
    from_date,
    to_date,
    api_key,
    page=1,
    page_size=50,
    order_by="oldest",
    section=None,
    timeout=30,
):
    """
    Fetch one page from The Guardian Content API.
    """
    params = {
        "api-key": api_key,
        "q": query,
        "from-date": from_date,     # YYYY-MM-DD
        "to-date": to_date,         # YYYY-MM-DD
        "page": page,
        "page-size": page_size,
        "order-by": order_by,
        "show-fields": "headline,bodyText,trailText",
    }

    if section:
        params["section"] = section

    response = requests.get(BASE_URL, params=params, timeout=timeout)
    response.raise_for_status()
    return response.json()


def normalize_guardian_article(article):
    """
    Extract date, title, content from Guardian API result.
    """
    fields = article.get("fields", {}) or {}

    return {
        "date": article.get("webPublicationDate", ""),
        "title": fields.get("headline", "") or article.get("webTitle", ""),
        "content": fields.get("bodyText", "") or fields.get("trailText", ""),
    }


def get_guardian_articles(
    query,
    from_date,
    to_date,
    api_key,
    page_size=50,
    sleep_between_requests=1,
    section=None,
):
    """
    Keep reading Guardian pages until all pages are collected.
    """
    rows = []
    page = 1
    total_pages = None

    while True:
        print(f"Fetching page {page}...")

        try:
            data = fetch_guardian_page(
                query=query,
                from_date=from_date,
                to_date=to_date,
                api_key=api_key,
                page=page,
                page_size=page_size,
                section=section,
            )
        except requests.RequestException as e:
            print(f"Request failed on page {page}: {e}")
            break

        response = data.get("response", {})
        results = response.get("results", [])

        if total_pages is None:
            total_pages = response.get("pages", 0)
            total_items = response.get("total", 0)
            print(f"Total results: {total_items}")
            print(f"Total pages: {total_pages}")

        if not results:
            print("No more results returned.")
            break

        rows.extend(normalize_guardian_article(article) for article in results)
        print(f"  Retrieved {len(results)} articles")

        if page >= total_pages:
            print("Last page reached.")
            break

        page += 1
        time.sleep(sleep_between_requests)

    df = pd.DataFrame(rows)

    if not df.empty:
        df = (
            df.drop_duplicates(subset=["date", "title", "content"])
              .sort_values("date")
              .reset_index(drop=True)
        )

    return df


# -----------------------------
# Example usage
# -----------------------------

QUERY = '"Trump"'
FROM_DATE = "2025-11-01"
TO_DATE = "2026-04-17"

df_guardian = get_guardian_articles(
    query=QUERY,
    from_date=FROM_DATE,
    to_date=TO_DATE,
    api_key=API_KEY,
    page_size=50,
    sleep_between_requests=1,
    section=None,   # e.g. "world", "politics", "us-news"
)

print(df_guardian.head())
print(f"\nGuardian articles found: {len(df_guardian)}")

df_guardian.to_csv("guardian_trump_articles.csv", index=False, encoding="utf-8")
print("\nSaved to guardian_trump_articles.csv")

Fetching page 1...
Total results: 7383
Total pages: 148
  Retrieved 50 articles
Fetching page 2...
  Retrieved 50 articles
Fetching page 3...
  Retrieved 50 articles
Fetching page 4...
  Retrieved 50 articles
Fetching page 5...
  Retrieved 50 articles
Fetching page 6...
  Retrieved 50 articles
Fetching page 7...
  Retrieved 50 articles
Fetching page 8...
  Retrieved 50 articles
Fetching page 9...
  Retrieved 50 articles
Fetching page 10...
  Retrieved 50 articles
Fetching page 11...
  Retrieved 50 articles
Fetching page 12...
  Retrieved 50 articles
Fetching page 13...
  Retrieved 50 articles
Fetching page 14...
  Retrieved 50 articles
Fetching page 15...
  Retrieved 50 articles
Fetching page 16...
  Retrieved 50 articles
Fetching page 17...
  Retrieved 50 articles
Fetching page 18...
  Retrieved 50 articles
Fetching page 19...
  Retrieved 50 articles
Fetching page 20...
  Retrieved 50 articles
Fetching page 21...
  Retrieved 50 articles
Fetching page 22...
  Retrieved 50 articles
Fetc